In [ ]:
# !pip install torch torchvision segmentation-models-pytorch scikit-learn matplotlib seaborn pandas

In [ ]:
import os, time, copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:

DATASET_PATH = Path("/Users/mutumihaela/Desktop/master/ic2/Dataset_BUSI_with_GT")

CLASE = ["benign", "malignant", "normal"]   # 0, 1, 2

print(f"Cale dataset: {DATASET_PATH.resolve()}")

In [ ]:
imagini, etichete = [], []

for idx, clasa in enumerate(CLASE):
    folder = DATASET_PATH / clasa
    for img_path in folder.glob("*.png"):
        if "_mask" in img_path.name:   
            continue
        imagini.append(img_path)
        etichete.append(idx)

print(f"Total imagini: {len(imagini)}")
for idx, clasa in enumerate(CLASE):
    print(f"  {clasa:<12}: {etichete.count(idx)}")

In [ ]:
# Cateva exemple
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row, clasa in enumerate(CLASE):
    imgs_clasa = [p for p, e in zip(imagini, etichete) if e == row]
    sample = random.sample(imgs_clasa, 3)
    for col, img_path in enumerate(sample):
        axes[row, col].imshow(Image.open(img_path), cmap="gray")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(clasa.upper(), fontweight="bold", loc="left")
plt.tight_layout()
plt.show()

In [37]:
#resize img
IMG_SIZE = 128

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

transform_test = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [38]:
class BUSIDataset(Dataset):
    def __init__(self, imagini, etichete, transform=None):
        self.imagini = imagini
        self.etichete = etichete
        self.transform = transform

    def __len__(self):
        return len(self.imagini)

    def __getitem__(self, idx):
        img = Image.open(self.imagini[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        # LONG pentru CrossEntropyLoss
        return img, torch.tensor(self.etichete[idx], dtype=torch.long)

In [ ]:
#split 70 / 15 / 15 - stratificat pe clase 

# test (15%)
img_tv, img_test, et_tv, et_test = train_test_split(
    imagini, etichete,
    test_size=0.15, stratify=etichete, random_state=SEED
)

# din restul (85%)--> separăm validare (17.6% din cele ramase)
img_train, img_val, et_train, et_val = train_test_split(
    img_tv, et_tv,
    test_size=0.176, stratify=et_tv, random_state=SEED
)

# dataset-urile
ds_train = BUSIDataset(img_train, et_train, transform=transform_train)
ds_val   = BUSIDataset(img_val,   et_val,   transform=transform_test)
ds_test  = BUSIDataset(img_test,  et_test,  transform=transform_test)

print(f"SPLIT 70/15/15:")
print(f"  Train: {len(ds_train)}")
print(f"  Val  : {len(ds_val)}")
print(f"  Test : {len(ds_test)}")

# proportii
print("\nDistribuție pe lot:")
for nume, labels in [("Train", et_train), ("Val", et_val), ("Test", et_test)]:
    procente = [labels.count(i)/len(labels)*100 for i in range(3)]
    print(f"  {nume:<6}: " + " | ".join(f"{c}={p:.0f}%" for c, p in zip(CLASE, procente)))

In [ ]:
#functie pentru DataLoaders
def creeaza_loadere(batch_size=16):
    loader_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=0)
    loader_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=0)
    loader_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False, num_workers=0)
    return loader_train, loader_val, loader_test

In [ ]:
BATCH_SIZES = [8, 16]
EPOCI_LIST  = [10, 25]
NR_CLASE    = 3

print(f"Modele     : MLP, U-Net, U-Net++")
print(f"Batch sizes: {BATCH_SIZES}")
print(f"Epoci      : {EPOCI_LIST}")
print(f"Total experimente: {3 * len(BATCH_SIZES) * len(EPOCI_LIST)}")

In [ ]:
# Exp 1-> MLP cu 6 Hidden Layers

class MLP(nn.Module):
    def __init__(self, input_size=3*128*128, nr_clase=3):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 1024), nn.ReLU(), nn.Dropout(0.3), 
            nn.Linear(1024, 512),        nn.ReLU(), nn.Dropout(0.3), 
            nn.Linear(512, 256),         nn.ReLU(), nn.Dropout(0.3),  
            nn.Linear(256, 128),         nn.ReLU(), nn.Dropout(0.2), 
            nn.Linear(128, 64),          nn.ReLU(), nn.Dropout(0.2),  
            nn.Linear(64, 32),           nn.ReLU(),                    
            nn.Linear(32, nr_clase)      # output-> 3 logits (fara softmax ce în crossentropyLoss)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

m = MLP()
print(f"MLP – parametri: {sum(p.numel() for p in m.parameters()):,}")
del m

In [ ]:
#Exp2 -> U-Net FULL (encoder + decoder + skip connections)

import segmentation_models_pytorch as smp

class UNetClasificare(nn.Module):
    def __init__(self, nr_clase=3):
        super().__init__()
        self.unet = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=3,
            classes=16   #output: feature maps cu 16 canale, la rezolutia originala
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16, nr_clase)
        )

    def forward(self, x):
        x = self.unet(x)   # (B, 16, 128, 128) – output U-Net complet
        x = self.gap(x)    # (B, 16, 1, 1)
        x = self.fc(x)     # (B, 3)
        return x

m = UNetClasificare()
print(f"U-Net Full – parametri: {sum(p.numel() for p in m.parameters()):,}")
del m

In [ ]:
#  U-Net++ FULL

class UNetPPClasificare(nn.Module):
    def __init__(self, nr_clase=3):
        super().__init__()
        self.unetpp = smp.UnetPlusPlus(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=3,
            classes=16
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16, nr_clase)
        )

    def forward(self, x):
        x = self.unetpp(x)
        x = self.gap(x)
        x = self.fc(x)
        return x

m = UNetPPClasificare()
print(f"U-Net++ Full – parametri: {sum(p.numel() for p in m.parameters()):,}")
del m

In [47]:
# functia de Antrenare
def antreneaza_model(model, loader_train, loader_val, nr_epoci=10, lr=0.0003):
    model = model.to(DEVICE)
    # ponderile pentru clase dezechilibrate
    counts = torch.tensor([437, 210, 133], dtype=torch.float)
    class_weights = (counts.sum() / (3 * counts)).to(DEVICE)
    criteriu = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    istoric = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None

    start = time.time()

    for epoca in range(1, nr_epoci + 1):
        # antrenare
        model.train()
        loss_train, corecte_train, total_train = 0, 0, 0

        for imgs, lbls in loader_train:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            preds = model(imgs)              # (B, 3) 
            loss = criteriu(preds, lbls)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_train += loss.item() * imgs.size(0)
            corecte_train += (preds.argmax(1) == lbls).sum().item()
            total_train += lbls.size(0)

        # validare
        model.eval()
        loss_val, corecte_val, total_val = 0, 0, 0

        with torch.no_grad():
            for imgs, lbls in loader_val:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                preds = model(imgs)
                loss = criteriu(preds, lbls)

                loss_val += loss.item() * imgs.size(0)
                corecte_val += (preds.argmax(1) == lbls).sum().item()
                total_val += lbls.size(0)

        # statistici
        avg_train_loss = loss_train / total_train
        avg_val_loss   = loss_val   / total_val
        acc_train      = corecte_train / total_train
        acc_val        = corecte_val   / total_val

        istoric["train_loss"].append(avg_train_loss)
        istoric["val_loss"].append(avg_val_loss)
        istoric["train_acc"].append(acc_train)
        istoric["val_acc"].append(acc_val)

        if acc_val > best_val_acc:
            best_val_acc = acc_val
            best_state = copy.deepcopy(model.state_dict())

        print(f"  Epoca {epoca:>3}/{nr_epoci} │ "
              f"Train: loss={avg_train_loss:.4f} acc={acc_train:.4f} │ "
              f"Val: loss={avg_val_loss:.4f} acc={acc_val:.4f}")

    timp_total = time.time() - start
    print(f"\n  ⏱️  Timp: {timp_total:.1f}s │ Best val_acc: {best_val_acc:.4f}")

    # cel mai bun model il salvam 
    model.load_state_dict(best_state)

    istoric["timp_antrenare"] = timp_total
    istoric["best_val_acc"] = best_val_acc

    return model, istoric

In [48]:
#functia de evaluare pe test
def evalueaza_model(model, loader_test, nume="Model"):
    model = model.to(DEVICE)
    model.eval()

    toate_pred, toate_lbls = [], []
    start = time.time()

    with torch.no_grad():
        for imgs, lbls in loader_test:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1).cpu().numpy()  # clasele prezise
            toate_pred.extend(preds)
            toate_lbls.extend(lbls.numpy())

    timp = time.time() - start

    toate_pred = np.array(toate_pred)
    toate_lbls = np.array(toate_lbls)

    acc = accuracy_score(toate_lbls, toate_pred)
    cm = confusion_matrix(toate_lbls, toate_pred, labels=[0, 1, 2])
    raport = classification_report(toate_lbls, toate_pred,
                                    target_names=CLASE,
                                    labels=[0, 1, 2],
                                    output_dict=True,
                                    zero_division=0)

    precision = raport["weighted avg"]["precision"]
    recall    = raport["weighted avg"]["recall"]
    f1        = raport["weighted avg"]["f1-score"]

    print(f"\n{'═' * 55}")
    print(f"  TEST: {nume}")
    print(f"{'═' * 55}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall   : {recall:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    print(f"  Timp     : {timp:.2f}s")
    print(f"  Confusion Matrix:")
    print(f"            {' '.join(c[:5].rjust(7) for c in CLASE)}")
    for i, clasa in enumerate(CLASE):
        print(f"  {clasa[:7]:<8} {' '.join(str(v).rjust(7) for v in cm[i])}")
    print(f"{'═' * 55}")

    return {
        "accuracy": acc, "precision": precision, "recall": recall, "f1": f1,
        "timp_inferenta": timp, "confusion_matrix": cm,
        "predictii": toate_pred, "etichete": toate_lbls
    }

In [49]:
#functii pentru grafice

def plot_learning_curves(istoric, nume="Model"):
    epoci = range(1, len(istoric["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(epoci, istoric["train_loss"], "b-o", label="Train", markersize=4)
    ax1.plot(epoci, istoric["val_loss"],   "r-o", label="Val",   markersize=4)
    ax1.set_title(f"{nume} – Loss", fontweight="bold")
    ax1.set_xlabel("Epocă"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epoci, istoric["train_acc"], "b-o", label="Train", markersize=4)
    ax2.plot(epoci, istoric["val_acc"],   "r-o", label="Val",   markersize=4)
    ax2.set_title(f"{nume} – Accuracy", fontweight="bold")
    ax2.set_xlabel("Epocă"); ax2.set_ylabel("Accuracy")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(cm, nume="Model"):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASE, yticklabels=CLASE)
    plt.xlabel("Predicție"); plt.ylabel("Real")
    plt.title(f"Confusion Matrix – {nume}", fontweight="bold")
    plt.tight_layout()
    plt.show()

In [54]:
#functia de rulat un experiment
def ruleaza_experiment(nume_model, model, batch_size, nr_epoci, lr=0.0003):
    eticheta = f"{nume_model} | batch={batch_size} | epoci={nr_epoci}"
    print(f"\n{'█' * 60}\n  ▶ {eticheta}\n{'█' * 60}")

    # DataLoaders
    loader_train, loader_val, loader_test = creeaza_loadere(batch_size)

    # antrenare
    model_antrenat, istoric = antreneaza_model(model, loader_train, loader_val, nr_epoci, lr)

    # evaluare
    rez_test = evalueaza_model(model_antrenat, loader_test, eticheta)

    # grafice
    plot_learning_curves(istoric, eticheta)
    plot_confusion_matrix(rez_test["confusion_matrix"], eticheta)

    return {
        "model":       nume_model,
        "batch_size":  batch_size,
        "nr_epoci":    nr_epoci,
        "train_acc":   istoric["train_acc"][-1],
        "val_acc":     istoric["val_acc"][-1],
        "test_acc":    rez_test["accuracy"],
        "precision":   rez_test["precision"],
        "recall":      rez_test["recall"],
        "f1":          rez_test["f1"],
        "timp_train":  istoric["timp_antrenare"],
        "timp_test":   rez_test["timp_inferenta"],
        "istoric":     istoric
    }

In [ ]:
toate_rezultatele = []

def creeaza_model(nume):
    if nume == "MLP":     return MLP()
    if nume == "U-Net":   return UNetClasificare()
    if nume == "U-Net++": return UNetPPClasificare()
    raise ValueError(f"Model necunoscut: {nume}")

MODELE = ["MLP", "U-Net", "U-Net++"]
print(f"Configurație:")
print(f"  Modele: {MODELE}")
print(f"  Batch : {BATCH_SIZES}")
print(f"  Epoci : {EPOCI_LIST}")
print(f"  Total : {len(MODELE) * len(BATCH_SIZES) * len(EPOCI_LIST)} experimente")

In [ ]:
#Experiment 1– MLP
for epoci in EPOCI_LIST:
    for batch in BATCH_SIZES:
        model = creeaza_model("MLP")
        rez = ruleaza_experiment("MLP", model, batch, epoci)
        toate_rezultatele.append(rez)
        del model

In [ ]:
#Experiment 2– U-Net
for epoci in EPOCI_LIST:
    for batch in BATCH_SIZES:
        model = creeaza_model("U-Net")
        rez = ruleaza_experiment("U-Net", model, batch, epoci)
        toate_rezultatele.append(rez)
        del model

In [ ]:
#Experiment 3– U-Net++
for epoci in EPOCI_LIST:
    for batch in BATCH_SIZES:
        model = creeaza_model("U-Net++")
        rez = ruleaza_experiment("U-Net++", model, batch, epoci)
        toate_rezultatele.append(rez)
        del model

df = pd.DataFrame([{k: v for k, v in r.items() if k != "istoric"}
                   for r in toate_rezultatele])

In [ ]:
#salvare csv
df = pd.DataFrame([{k: v for k, v in r.items() if k != "istoric"}
                   for r in toate_rezultatele])

# Rotunjire
for col in ["train_acc", "val_acc", "test_acc", "precision", "recall", "f1"]:
    df[col] = df[col].round(4)
df["timp_train"] = df["timp_train"].round(1)
df["timp_test"] = df["timp_test"].round(2)

df.to_csv("rezultate.csv", index=False)
df

In [ ]:
#test accuracy vs nr. epoci- grafic
fig, ax = plt.subplots(figsize=(10, 6))
culori_model = {"MLP": "#2196F3", "U-Net": "#4CAF50", "U-Net++": "#FF9800"}

for model_name in MODELE:
    df_m = df[df["model"] == model_name]
    pivot = df_m.pivot_table(index="nr_epoci", values="test_acc", aggfunc="mean")
    ax.plot(pivot.index, pivot["test_acc"], "-o",
            label=model_name, linewidth=2, markersize=10,
            color=culori_model[model_name])

ax.set_title("Test Accuracy vs Nr. Epoci", fontweight="bold", fontsize=13)
ax.set_xlabel("Nr. Epoci"); ax.set_ylabel("Test Accuracy")
ax.set_xticks(EPOCI_LIST)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# test accuracy vs batch size- grafic
fig, ax = plt.subplots(figsize=(10, 6))

for model_name in MODELE:
    df_m = df[df["model"] == model_name]
    pivot = df_m.pivot_table(index="batch_size", values="test_acc", aggfunc="mean")
    ax.plot(pivot.index, pivot["test_acc"], "-o",
            label=model_name, linewidth=2, markersize=10,
            color=culori_model[model_name])

ax.set_title("Test Accuracy vs Batch Size", fontweight="bold", fontsize=13)
ax.set_xlabel("Batch Size"); ax.set_ylabel("Test Accuracy")
ax.set_xticks(BATCH_SIZES)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#timp antrenare per model- grafic
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# timp antrenare mediu per model
timp_train_mean = df.groupby("model")["timp_train"].mean()
timp_test_mean  = df.groupby("model")["timp_test"].mean()

bars1 = ax1.bar(timp_train_mean.index, timp_train_mean.values,
                color=[culori_model[m] for m in timp_train_mean.index],
                edgecolor="black")
ax1.set_title("Timp Antrenare Mediu (s)", fontweight="bold")
ax1.set_ylabel("Secunde")
ax1.grid(True, alpha=0.3, axis="y")
for bar, v in zip(bars1, timp_train_mean.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{v:.0f}s", ha="center", va="bottom", fontweight="bold")

bars2 = ax2.bar(timp_test_mean.index, timp_test_mean.values,
                color=[culori_model[m] for m in timp_test_mean.index],
                edgecolor="black")
ax2.set_title("Timp Inferență Test Mediu (s)", fontweight="bold")
ax2.set_ylabel("Secunde")
ax2.grid(True, alpha=0.3, axis="y")
for bar, v in zip(bars2, timp_test_mean.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{v:.2f}s", ha="center", va="bottom", fontweight="bold")

plt.suptitle("⏱️ Comparație Timpi", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#test accuracy per configuratie
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, model_name in zip(axes, MODELE):
    df_m = df[df["model"] == model_name]
    pivot = df_m.pivot_table(index="batch_size", columns="nr_epoci", values="test_acc")

    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd",
                vmin=0.3, vmax=1.0, ax=ax,
                cbar=True, linewidths=1, linecolor="white")
    ax.set_title(f"{model_name}", fontweight="bold")
    ax.set_xlabel("Nr. Epoci")
    ax.set_ylabel("Batch Size")

plt.suptitle(" Test Accuracy – Heatmap per Model", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#comparatie finala – Bar Chart
fig, ax = plt.subplots(figsize=(11, 6))

x = np.arange(len(MODELE))
latime = 0.2

# 4 bare per model: combinatiile (batch, epoci)
configuratii = [(b, e) for e in EPOCI_LIST for b in BATCH_SIZES]
culori_config = ["#90CAF9", "#42A5F5", "#1E88E5", "#0D47A1"]

for i, (b, e) in enumerate(configuratii):
    vals = []
    for m in MODELE:
        row = df[(df["model"] == m) & (df["batch_size"] == b) & (df["nr_epoci"] == e)]
        vals.append(row["test_acc"].values[0] if len(row) > 0 else 0)
    bars = ax.bar(x + i * latime, vals, latime,
                  label=f"batch={b}, epoci={e}", color=culori_config[i])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{v:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_title("🏆 Comparație Finală: Test Accuracy per Model și Configurație",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Model"); ax.set_ylabel("Test Accuracy")
ax.set_xticks(x + 1.5 * latime)
ax.set_xticklabels(MODELE)
ax.legend(fontsize=10, loc="upper left")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()